### Delta log P area under the ablation curve

In [1]:
import json
import pandas as pd

# Load results
with open("../results/master_results.json") as f:
    data = json.load(f)
     
# Convert to DataFrame (each experiment = row, metrics = columns)
df = pd.DataFrame(data).T
df.index.name = "experiment"
df = df.reset_index()

# Display as table
df

,experiment,top_k_fraction,avg_delta,variance_delta,sem_delta,accuracy,num_correct,total
0,Llama-3.2-3B__Temperature_lambada_top0.05,0.05,-4.954160,20.151400,0.259174,0.726667,218.0,300.0
1,Llama-3.2-3B__Temperature_lambada_top0.1,0.10,-6.823024,18.421273,0.247799,0.726667,218.0,300.0
2,Llama-3.2-3B__Temperature_lambada_top0.2,0.20,-8.917719,18.778210,0.250188,0.726667,218.0,300.0
3,Llama-3.2-3B__Semantic_lambada_top0.05,0.05,-5.050690,18.781744,0.250212,0.726667,218.0,300.0
4,Llama-3.2-3B__Semantic_lambada_top0.1,0.10,-6.770932,18.357299,0.247368,0.726667,218.0,300.0
5,Llama-3.2-3B__Semantic_lambada_top0.2,0.20,-9.157999,15.859307,0.229923,0.726667,218.0,300.0
6,Llama-3.2-3B__gradient_x_input_lambada_top0.05,0.05,-4.604373,17.710232,0.242969,0.726667,218.0,300.0
7,Llama-3.2-3B__gradient_x_input_lambada_top0.1,0.10,-6.364562,18.214974,0.246407,0.726667,218.0,300.0
8,Llama-3.2-3B__gradient_x_input_lambada_top0.2,0.20,-8.318417,16.888754,0.237267,0.726667,218.0,300.0
9,Llama-3.2-3B__IG_lambada_top0.05,0.05,-1.135767,5.033536,0.129532,0.726667,218.0,300.0


In [2]:
# Filter lambada experiments and pivot by drop fraction
lambada_data = {k: v for k, v in data.items() if "lambada" in k.lower() and "3B" in k}

def fmt_val_plus_sem(entry):
    """Format avg_delta ± sem_delta."""
    if entry is None:
        return None
    avg = entry.get("avg_delta")
    sem = entry.get("sem_delta")
    if avg is None:
        return None
    if sem is not None:
        return f"{avg:.2f} ± {sem:.2f}"
    return f"{avg:.2f}"

# Build pivoted table: rows = drop fraction (0.05, 0.1, 0.2), columns = method type
rows = []
for frac in [0.05, 0.1, 0.2]:
    frac_str = f"top{frac}"
    row = {"method": f"{int(frac*100)}%"}
    
    # random drop
    random_key = next((k for k in lambada_data if frac_str in k and "random_ablation" in k), None)
    row["random"] = fmt_val_plus_sem(lambada_data[random_key] if random_key else None)

    grad_key = next((k for k in lambada_data if "IG" in k and frac_str in k), None)
    row["Integrated Grads"] = fmt_val_plus_sem(lambada_data[grad_key] if grad_key else None)
    
    grad_key = next((k for k in lambada_data if "gradient_x_input" in k and frac_str in k), None)
    row["Input x Grad"] = fmt_val_plus_sem(lambada_data[grad_key] if grad_key else None)
                        
    # top k with temperature scope
    sem_key = next((k for k in lambada_data if "Temperature" in k and frac_str in k and "random_drop" not in k), None)
    row["Temperature Scope"] = fmt_val_plus_sem(lambada_data[sem_key] if sem_key else None)

    # top k with semantic scope
    sem_key = next((k for k in lambada_data if "Semantic" in k and frac_str in k and "random_drop" not in k), None)
    row["Semantic Scope"] = fmt_val_plus_sem(lambada_data[sem_key] if sem_key else None)
    

    rows.append(row)

lambada_table = pd.DataFrame(rows).set_index("method").T
lambada_table

method,5%,10%,20%
random,-0.62 ± 0.10,-1.04 ± 0.11,-2.48 ± 0.19
Integrated Grads,-1.14 ± 0.13,-3.29 ± 0.21,-6.57 ± 0.25
Input x Grad,-4.60 ± 0.24,-6.36 ± 0.25,-8.32 ± 0.24
Temperature Scope,-4.95 ± 0.26,-6.82 ± 0.25,-8.92 ± 0.25
Semantic Scope,-5.05 ± 0.25,-6.77 ± 0.25,-9.16 ± 0.23


In [5]:
print(lambada_table.to_markdown())

|                   | 5%           | 10%          | 20%          |
|:------------------|:-------------|:-------------|:-------------|
| random            | -0.50 ± 0.08 | -1.09 ± 0.13 | -2.53 ± 0.18 |
| Integrated Grads  | -1.14 ± 0.13 | -3.29 ± 0.21 | -6.57 ± 0.25 |
| Input x Grad      | -4.60 ± 0.24 | -6.36 ± 0.25 | -8.32 ± 0.24 |
| Temperature Scope | -4.95 ± 0.26 | -6.82 ± 0.25 | -8.92 ± 0.25 |
| Semantic Scope    | -5.05 ± 0.25 | -6.77 ± 0.25 | -9.16 ± 0.23 |


### Most Influential Token

In [9]:

# Filter entries with "loo_rank" (influence ranking vs LOO comparison)
loo_rank_data = {k: v for k, v in data.items() if "loo_rank" in k}

# Map raw method names to display names (same as lambada table)
METHOD_DISPLAY = {
    "Random": "random",
    "Temperature": "Temperature Scope",
    "gradient_x_input": "Input x Grad",
    "Semantic": "Semantic Scope",
    "IG": "Integrated Grads",
    "Fisher": "Fisher Scope",
    
}

# Build lookup: label key → mean_ranking_pct ± SEM string
def fmt_loo_val(entry):
    mean_pct = entry.get("mean_ranking_pct")
    sem_pct = entry.get("sem_mean_ranking_pct")
    if mean_pct is not None and sem_pct is not None:
        return f"{mean_pct:.1f} ± {sem_pct:.1f}%"
    elif mean_pct is not None:
        return f"{mean_pct:.1f}%"
    return None

method_to_val = {}
for label, entry in loo_rank_data.items():
    parts = label.split("__")
    raw_method = parts[1].replace("_lambada_loo_rank", "") if len(parts) >= 2 else label
    display_method = METHOD_DISPLAY.get(raw_method, raw_method)
    method_to_val[display_method] = fmt_loo_val(entry)

# Build table with same row order as lambada: random, Temperature Scope, Semantic Scope, Gradient Input
# row_order = ["random", "Semantic Scope", "Gradient Input", "Temperature Scope"]
row_order = ["random","Integrated Grads", "Semantic Scope", "Input x Grad", "Temperature Scope", "Fisher Scope"]
loo_table = pd.DataFrame(
    {"average ranking": [method_to_val.get(m) for m in row_order]},
    index=row_order,
)
loo_table.index.name = "method"
loo_table

,average ranking
method,
random,51.3 ± 2.9%
Integrated Grads,19.2 ± 2.6%
Semantic Scope,7.0 ± 1.2%
Input x Grad,6.8 ± 1.2%
Temperature Scope,6.5 ± 1.1%
Fisher Scope,5.9 ± 1.1%


In [ ]:
print(loo_table.to_markdown())

| method            | average ranking   |
|:------------------|:------------------|
| random            | 51.3 ± 2.9%       |
| Integrated Grads  | 19.2 ± 2.6%       |
| Semantic Scope    | 7.0 ± 1.2%        |
| Input x Grad      | 6.8 ± 1.2%        |
| Temperature Scope | 6.5 ± 1.1%        |
| Fisher Scope      | 5.9 ± 1.1%        |
